# POLG burden analysis

- **Project**: Multi-ancestry analysis of POLG variants in Parkinson’s disease
- **Last Update:** APRIL-2026

## Define paths

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for Dataframe functionality
import pandas as pd

import subprocess

# Numpy for basics
import numpy as np

# Use pathlib for file path manipulation
import pathlib

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

import re

### Install packages

In [ ]:
%%capture
%%bash

#To install plink 1.9
cd /home/jupyter/
if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter

    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

#chmod plink 1.9 to ensure permission to run the program
chmod u+x /home/jupyter/plink1.9

In [ ]:
%%capture
%%bash

#To install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [ ]:
%%bash

#chmod plink 2 to ensure permission to run the program
chmod u+x /home/jupyter/plink2

In [ ]:
%%bash

#To install ANNOVAR after registration on the annovar website - https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/annovar; then

echo "annovar is already installed in /home/jupyter/"
else
echo "annovar is not installed"
cd /home/jupyter/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

In [ ]:
!wget http://www.openbioinformatics.org/annovar/download/hg38_clinvar_20240908.txt.gz

In [ ]:
%%bash

#To download resources for annotation

cd /home/jupyter/annovar/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20250721 humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/ 
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_genome humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_exome humandb/

In [ ]:
%%capture
%%bash

#To install RVTESTS
if test -e /home/jupyter/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /home/jupyter/rvtests
cd /home/jupyter/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 

tar -zxvf rvtests_linux64.tar.gz
fi

In [ ]:
#chmod to ensure permission to run rvtests
! chmod 777 /home/jupyter/rvtests/executable/rvtest

In [ ]:
%%bash

#To ensure rvtest has been installed successfully
/home/jupyter/rvtests/executable/rvtest --help

In [ ]:
WORK_DIR = "/path/to/working/directory"

In [ ]:
#To get refFlat.txt
!cd {WORK_DIR}
!wget http://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz
!gunzip refFlat.txt.gz

### POLG variant classification and grouping

In [ ]:
ancestries = {'AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE'}

In [ ]:
!mkdir {WORK_DIR}/BURDEN_WGS

In [ ]:
#To make directories for each ancestry
ancestries = {'AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE'}

for ancestry in ancestries:
    !mkdir {WORK_DIR}/BURDEN_WGS/{ancestry}

### Prepare files for annotation

In [ ]:
#To extract POLG region using plink
for ancestry in ancestries:
    
    ! /home/jupyter/plink2 \
    --pfile {REL11_PATH}/wgs/deepvariant_joint_calling/plink/{ancestry}/chr15_{ancestry}_release11 \
    --chr 15 \
    --from-bp 89305198 \
    --to-bp 89334861 \
    --mac 2 \
    --hwe 1e-5 0 \
    --make-pgen \
    --out {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG

In [ ]:
ancestries = {'AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE'}
for ancestry in ancestries:
        
    ## Turn binary files into VCF
    ! /home/jupyter/plink2 \
    --pfile {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG \
    --recode vcf id-paste=iid \
    --out {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG

In [ ]:
#Bgzip and Tabix - to zip and index the VCF files
ancestries = {'AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE'}
for ancestry in ancestries:    
    ! bgzip -f {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.vcf
    ! tabix -f -p vcf {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.vcf.gz 

### Annotate

In [ ]:
## annotate using ANNOVAR
ancestries = {'AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE'}

for ancestry in ancestries:
        
    ! perl /home/jupyter/annovar/table_annovar.pl {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.vcf.gz /home/jupyter/annovar/humandb/ -buildver hg38 \
    -out {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.annovar \
    -remove -protocol refGene,clinvar_20250721,dbnsfp47a,gnomad41_genome,gnomad41_exome \
    -operation g,f,f,f,f \
    --nopolish \
    -nastring . \
    -vcfinput

### Extract variant sets

In [ ]:
for ancestry in ancestries:      

    gene = pd.read_csv(f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.annovar.hg38_multianno.txt', sep = '\t')
    gene = gene[gene['Gene.refGene'] == 'POLG']

    # Convert columns to numeric safely
    for col in ['gnomad41_genome_fafmax_faf95_max','gnomad41_exome_fafmax_faf95_max','CADD_phred', 'REVEL_score', 'PrimateAI_score']:
        if col in gene.columns:
            gene[col] = pd.to_numeric(gene[col], errors='coerce')

    # Rarity by gnomAD
    maf_1  = (gene['gnomad41_genome_fafmax_faf95_max'] < 0.01) & (gene['gnomad41_exome_fafmax_faf95_max'] < 0.01)
    maf_01 = (gene['gnomad41_genome_fafmax_faf95_max'] < 0.001) & (gene['gnomad41_exome_fafmax_faf95_max'] < 0.001)

    # LoF definition
    lof = (
        (gene['Func.refGene'] == 'splicing') |
        ((gene['Func.refGene'] == 'exonic') & (
            (gene['ExonicFunc.refGene'] == 'stopgain') |
            (gene['ExonicFunc.refGene'] == 'stoploss') |
            (gene['ExonicFunc.refGene'] == 'startloss') |
            (gene['ExonicFunc.refGene'] == 'frameshift deletion') |
            (gene['ExonicFunc.refGene'] == 'frameshift insertion')
        ))
    )

    # Missense (nonsynonymous SNVs)
    missense = (
        (gene['Func.refGene'] == 'exonic') &
        (gene['ExonicFunc.refGene'] == 'nonsynonymous SNV')
    )

    # Damaging missense
    cadd_20 = gene['CADD_phred'] >= 20
    revel_06 = gene['REVEL_score'] >= 0.644
    prim_08 = gene['PrimateAI_score'] >= 0.8

    poly_d = gene['Polyphen2_HDIV_pred'] == 'D'
    sift_d = gene['SIFT4G_pred'] == 'D'

    # 2-of-4; also accept strong REVEL or PrimateAI
    prediction = (revel_06.fillna(False).astype(int) +
                prim_08.fillna(False).astype(int) +
                poly_d.fillna(False).astype(int) +
                sift_d.fillna(False).astype(int))

    damaging_missense = missense & cadd_20 & ((prediction >= 2) | revel_06 | prim_08)

    # ClinVar strict P/LP (exclude obvious conflicts/benign)
    clnsig = gene.get('CLNSIG', pd.Series(index=gene.index)).fillna('').str.lower()
    clinvar_strict = (clnsig.str.contains('pathogenic') & ~clnsig.str.contains('conflict') & ~clnsig.str.contains('benign'))

    filtered_variants1 = gene[maf_1]                                # rare
    filtered_variants2 = gene[maf_01]                               # ultra-rare
    filtered_variants3 = gene[maf_1 & missense]                     # rare missense
    filtered_variants4 = gene[maf_01 & missense]                    # ultra-rare + missense
    filtered_variants5 = gene[maf_1 & lof]                          # rare LoF
    filtered_variants6 = gene[maf_01 & lof]                         # ultra-rare LoF
    filtered_variants7 = gene[maf_1 & damaging_missense]            # rare damaging missense
    filtered_variants8 = gene[maf_01 & damaging_missense]           # ultra-rare damaging missense
    filtered_variants9 = gene[maf_1 & clinvar_strict]               # rare ClinVar P/LP
    filtered_variants10 = gene[maf_01 & clinvar_strict]             # ultra-rare ClinVar P/LP
                                  
    # SAVE GROUPS IN PLINK FORMAT
    def save_plink_range(df: pd.DataFrame, outpath: str) -> None:
        """
        Save as PLINK --extract range file: CHR START END ID (no header).
        """
        out = df[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        out = out.dropna(subset=['Chr', 'Start', 'End', 'Gene.refGene']).drop_duplicates()
        out.to_csv(outpath, sep="\t", index=False, header=False)

    save_plink_range(filtered_variants1, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf1.variantstoKeep.txt')
    save_plink_range(filtered_variants2, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf01.variantstoKeep.txt')
    save_plink_range(filtered_variants3, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf1_M.variantstoKeep.txt')
    save_plink_range(filtered_variants4, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf01_M.variantstoKeep.txt')
    save_plink_range(filtered_variants5, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf1_LOF.variantstoKeep.txt')
    save_plink_range(filtered_variants6, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf01_LOF.variantstoKeep.txt')
    save_plink_range(filtered_variants7, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf1_damagingMissense.variantstoKeep.txt')
    save_plink_range(filtered_variants8, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf01_damagingMissense.variantstoKeep.txt')
    save_plink_range(filtered_variants9, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf1_ClinVarPLP.variantstoKeep.txt')
    save_plink_range(filtered_variants10, f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.maf01_ClinVarPLP.variantstoKeep.txt')

### Create covariate files 

In [ ]:
key = pd.read_csv("f{REL11_PATH}/clinical_data/master_key_release11_final_vwb.csv", low_memory=False)
key

In [ ]:
#To subset master key to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label','wgs_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'age_of_onset':'AAO'}, inplace = True)

In [ ]:
#To tidy ancestry label for NBA and WGS samples
key["label"] = key["nba_label"].combine_first(key["wgs_label"])
key = key.drop(columns=["nba_label", "wgs_label"])
key

In [ ]:
#To tidy and create demographic subfiles from master key for each ancestry 
ancestries = {'AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE'}

for ancestry in ancestries:
    
    print(f'WORKING ON: {ancestry}')
    
    ## Subset to keep ancestry of interest 
    ancestry_key = key[key['label']==ancestry].copy()
    ancestry_key.reset_index(drop=True)
    
    # Convert phenotype to binary (1/2)
    ## Assign conditions so case=2 and controls=1, and -9 otherwise (matching PLINK convention)
    # PD = 2; control = 1
    pheno_mapping = {"PD": 2, "Control": 1}
    ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')

    # Check value counts of pheno
    ancestry_key['PHENO'].value_counts(dropna=False)
    
    ## Get the PCs
    pcs = pd.read_csv(f'{REL11_PATH}/wgs/deepvariant_joint_calling/pcs/{ancestry}/{ancestry}_release11.eigenvec', sep='\t')
    
    #Select just first 5 PCs
    selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
    pcs = pd.DataFrame(data=pcs.iloc[:, 1:7].values, columns=selected_columns)

    # Drop the first row (since it's now the column names)
    pcs = pcs.drop(0)

    # Reset the index to remove any potential issues
    pcs = pcs.reset_index(drop=True)
    
    # Check size
    print(f'PCs: {pcs.shape}')

    # Check value counts of SEX
    sex_og_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - original:\n {sex_og_values.to_string()}')
    
    # Convert sex to binary (1/2)
    ## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
    # Female = 2; Male = 1
    sex_mapping = {"Female": 2, "Male": 1}
    ancestry_key['SEX'] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')
    
    # Check value counts of SEX after recoding
    sex_recode_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - recoded:\n{sex_recode_values.to_string()}')
    
    ## Make covariate file
    df = pd.merge(ancestry_key,pcs, on='IID')
    print(f'Check columns for covariate file: {df.columns}')

    # Load information about related individuals in the ancestry analyzed
    related_df = pd.read_csv(f'{REL11_PATH}/meta_data/related_samples/{ancestry}_release11_vwb.related')
    print(f'Related individuals: {related_df.shape}')
    
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])
    related_list = [re.sub(r'_s.*$', '', iid) for iid in related_list]
    print(f'Number of related IIDs in dataset before filtering: {df["IID"].isin(related_list).sum()}')
    
    # Check value counts of related and remove only one related individual
    df = df[~df["IID"].isin(related_list)]

    # Check size
    print(f'Unrelated individuals: {df.shape}')
    
    #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
    #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
    #Uppercase column name is ok
    #See https://zhanxw.github.io/rvtests/#phenotype-file
    df['FID'] = 0
    df['FATID'] = 0
    df['MATID'] = 0

    ## Clean up and keep columns we need 
    final_df = df[['FID','IID', 'FATID', 'MATID', 'SEX', 'AGE','AAO', 'PHENO','PC1', 'PC2', 'PC3', 'PC4', 'PC5']].copy()

    ##DO NOT replace missing values with -9 as this is misinterpreted by RVtests - needs to be nonnumeric
    #Leave missing values as NA
    
    #Check number of PD cases missing age
    pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE'].isna())]
    print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')
    
    #Check number of controls missing age
    control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE'].isna())]
    print(f'Number of controls missing age: {control_missAge.shape[0]}')

    ## Make file of sample IDs to keep 
    samples_toKeep = final_df[['FID', 'IID']].copy()
    samples_toKeep.columns = ['#FID','IID']
    
    samplestokeep_path = pathlib.Path(pathlib.Path.home(), f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}.samplestokeep')
    
    # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
    if not samplestokeep_path.parent.exists():
        !mkdir -p {samplestokeep_path.parent}
        print(f'Created {samplestokeep_path.parent}')
    
    samples_toKeep.to_csv(samplestokeep_path, sep = '\t', index=False)

    finaldf_path = pathlib.Path(pathlib.Path.home(), f'{WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_covariate_file.txt')
    
    # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
    if not finaldf_path.parent.exists():
        !mkdir -p {finaldf_path.parent}
        print(f'Created {finaldf_path.parent}')
    
    final_df.to_csv(finaldf_path, sep = '\t', na_rep='NA', index=False)

In [ ]:
%%bash
#change FID to #FID
for ancestry in AFR AJ AMR EAS EUR MDE; do
  file=f"{WORK_DIR}/BURDEN_WGS/${ancestry}/${ancestry}_covariate_file.txt"
  sed -i '1s/^FID/#FID/' "$file"
done


In [ ]:
%%bash
#change #FID to 0
for ancestry in AFR AJ AMR EAS EUR MDE; do
  file=f"{WORK_DIR}/BURDEN_WGS/${ancestry}/${ancestry}_POLG.psam"
  echo "Updating $file ..."
  awk 'BEGIN{OFS="\t"} 
       NR==1 {
         for (i=1; i<=NF; i++) if ($i=="#FID") fid=i
         print
         next
       }
       {
         $fid=0
         print
       }' "$file" > "${file}.tmp" && mv "${file}.tmp" "$file"
done


In [ ]:
ancestries = ['AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE']
#Loop over all the ancestries to update phenotype column
for ancestry in ancestries:
        
        #Update phenotype
        !/home/jupyter/plink2 \
        --pfile {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG \
        --keep {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}.samplestokeep \
        --pheno {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_covariate_file.txt \
        --pheno-name PHENO \
        --make-pgen \
        --out {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG_newpheno

In [ ]:
%%bash
#To generate a covarite file that match columns needed for RVtests 
for ancestry in AFR AJ AMR EAS EUR MDE; do
  infile=f"{WORK_DIR}/BURDEN_WGS/${ancestry}/${ancestry}_covariate_file.txt"
  outfile=f"{WORK_DIR}/BURDEN_WGS/${ancestry}/${ancestry}_covariate_file_fixed.txt"
  echo "Fixing header in $infile → $outfile ..."
  sed '1s/^#FID/FID/' "$infile" > "$outfile"
done

### Extract variant groups and perform burden analysis

In [ ]:
variant_classes = ['maf1','maf01','maf1_M','maf01_M','maf1_LOF', 'maf01_LOF','maf1_damagingMissense','maf01_damagingMissense','maf1_ClinVarPLP','maf01_ClinVarPLP']

#Loop over all the ancestries and the variant classes
for ancestry in ancestries:      
    for variant_class in variant_classes:
                
        # Print the command to be executed (for debugging purposes)
        print(f'Running plink to extract {variant_class} variants for ancestry: {ancestry}')
        
        #Extract relevant variants
        ! /home/jupyter/plink2 \
        --pfile {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG_newpheno \
        --extract range {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.{variant_class}.variantstoKeep.txt \
        --recode vcf-iid \
        --out {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.{variant_class}
        
        # Print the command to be executed (for debugging purposes)
        print(f'Running bgzip and tabix for {variant_class} variants for ancestry: {ancestry}')
        
        ## Bgzip and Tabix (zip and index the file)
        ! bgzip -f {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.{variant_class}.vcf
        ! tabix -f -p vcf {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.{variant_class}.vcf.gz

In [ ]:
ancestries = ['AFR', 'AJ', 'AMR', 'EAS', 'EUR', 'MDE']

#To run RVtests
variant_classes = ['maf1','maf01','maf1_M','maf01_M','maf1_LOF', 'maf01_LOF','maf1_damagingMissense','maf01_damagingMissense','maf1_ClinVarPLP','maf01_ClinVarPLP']
for ancestry in ancestries:      
    for variant_class in variant_classes:
                
        # Print the command to be executed (for debugging purposes)
        print(f'Running RVtests for {variant_class} variants for ancestry: {ancestry}')
        
        ## RVtests with covariates 
        #Make sure the pheno and covariate file starts with the first 5 columns: fid, iid, fatid, matid, sex
        ! /home/jupyter/rvtests/executable/rvtest --noweb --hide-covar \
        --out {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.burden.{variant_class} \
        --kernel skat,skato \
        --inVcf {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_POLG.{variant_class}.vcf.gz \
        --pheno {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_covariate_file_fixed.txt \
        --pheno-name PHENO \
        --gene POLG \
        --geneFile {WORK_DIR}/refFlat.txt \
        --covar {WORK_DIR}/BURDEN_WGS/{ancestry}/{ancestry}_covariate_file_fixed.txt \
        --covar-name SEX,AGE,PC1,PC2,PC3,PC4,PC5